# M4_siren_pinnsformer
Spectral Variant: PINNsFormer with Periodic sin Activation


In [ ]:
using Plots, Flux, NNlib, LinearAlgebra, Statistics, Random, CSV, DataFrames, Optim, Printf, JSON

include("architectures.jl")
include("loss_and_optimization.jl")
include("benchmark_runner.jl")
include("paper_plots.jl")


In [ ]:
# 1. Load Ground-Truth Hodgkin-Huxley Synthetic Dataset
file_path = raw"C:\nirbhay\Downloads\NeuroPinnsFormmer-attention-that-neurons-needs\Synthetic_Data\HH_ground_truth_synthetic_data.csv"
HH_data = CSV.read(file_path, DataFrame)
checkpoint_dir = raw"C:\nirbhay\Downloads\NeuroPinnsFormmer-attention-that-neurons-needs\detailed_pinnsformmer_reports\checkpoints"


In [ ]:
# 2. Instantiate Model (M4) with Deterministic Seed = 42
Random.seed!(42)
model = PINNsFormer(; d_model=32, k=10, n_heads=4, depth=1, act_type=:siren)
println("Model Parameter Count: ", count_parameters(model))


In [ ]:
# 3. Execute Dual-Stage Optimization (1000 Adam + 200 L-BFGS) & Save Checkpoints
dataloader, _, _ = prepare_dataloader(HH_data, 10; batch_size=64)

trained_model, wall_time, history, best_weights, milestones = train_model_single_run(
    model, dataloader; 
    adam_epochs=1000, 
    lbfgs_epochs=200, 
    use_ntk=true,
    seed=42
)

metrics = evaluate_relative_l2_error(trained_model, HH_data)

# AUTOMATIC WEIGHT SERIALIZATION: Save Final, Best, and Intermediate Milestone Checkpoints
save_model_checkpoints("M4", 42, trained_model, best_weights, milestones, history, metrics, HH_data, checkpoint_dir)

println("--> Training Finished in $(round(wall_time, digits=2)) seconds")
println("--> Relative L2 Error: $(round(metrics.l2_total, digits=6))")
println("--> Voltage MSE (V)  : $(round(metrics.mse_V, digits=6))")


In [ ]:
# 4. Instant Weight Injection & Visual Audit (NO RE-TRAINING NEEDED)
# Reload stored parameters from disk to verify instant interpolation & prediction
restored_model, checkpoint_data = load_model_checkpoint("M4", 42; checkpoint_type="best")
restored_metrics = evaluate_relative_l2_error(restored_model, HH_data)

configure_paper_style()
t = HH_data.t
p_v = plot(t, HH_data.V, color=:black, line=:dash, label="Ground Truth (Radau)")
plot!(t, restored_metrics.V_pred, color=:crimson, label="Restored M4 Prediction")
title!("Restored Checkpoint Membrane Potential V(t)")
ylabel!("Voltage (mV)")

p_g = plot(t, HH_data.m, color=:black, line=:dash, label="GT m")
plot!(t, restored_metrics.m_pred, color=:dodgerblue, label="Restored m")
plot!(t, restored_metrics.h_pred, color=:darkorange, label="Restored h")
plot!(t, restored_metrics.n_pred, color=:forestgreen, label="Restored n")
title!("Restored Checkpoint Ion Channel Kinetics")
xlabel!("Time (ms)")
ylabel!("Gating Probability")

dashboard = plot(p_v, p_g, layout=grid(2, 1, heights=[0.5, 0.5]), link=:x, size=(800, 600))
display(dashboard)
